# Water Demand Forecasting

## Project Overview

Forecasts **daily water demand** (megalitres) over a 14-day horizon. Synthetic data spans 2 years with weekly patterns, summer peaks, and holiday dips.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily water consumption, predict the next 14 days. Water utilities need demand forecasts for treatment plant scheduling, reservoir management, and infrastructure planning.

## Why This Project Matters

Water is a critical resource. Over-production wastes energy and chemicals; under-production causes supply disruptions. Accurate forecasting optimizes treatment operations, reduces costs, and ensures reliable supply to communities.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily water demand
- Weekly pattern (higher weekdays, slight weekend dip)
- Summer peak (irrigation, cooling, outdoor use)
- Holiday dips (industrial slowdown)
- Gradual upward trend (population growth)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `water_ml` | Daily water demand (megalitres) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'water_ml'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 500 + 0.08 * t  # gradual population growth
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 15
    else: weekly[i] = -20

# Summer peak (outdoor watering, pools)
seasonal = 60 * np.sin(2 * np.pi * (t - 100) / 365)

# Holiday dips
holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if (m == 12 and d >= 24) or (m == 1 and d <= 2): holiday[i] = -40

noise = np.random.normal(0, 15, N_POINTS)
water_ml = base + weekly + seasonal + holiday + noise
water_ml = np.maximum(water_ml, 200).round(1)

df = pd.DataFrame({'date': dates, 'water_ml': water_ml})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,water_ml
0,2022-01-01,388.1
1,2022-01-02,378.5
2,2022-01-03,465.3
3,2022-01-04,478.4
4,2022-01-05,452.0


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('water_ml Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("water_ml Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

water_ml Statistics:
count    730.000000
mean     532.787397
std       53.019115
min      378.500000
25%      495.250000
50%      532.350000
75%      574.775000
max      649.500000
Name: water_ml, dtype: float64

CV: 0.100
Skewness: -0.180


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       34.0 | RMSE:       42.3 | MAPE:  7.51%
Seasonal Naive (7)             | MAE:       26.8 | RMSE:       33.7 | MAPE:  5.79%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                           Adjusted R-Squared   R-Squared        RMSE  Time Taken
Model                                                                            
KernelRidge                       8566.909871 -657.916144  545.991121    0.031190
MLPRegressor                      3129.515304 -239.655023  329.965073    0.543283
GaussianProcessRegressor          2195.292412 -167.791724  276.341481    0.084977
QuantileRegressor                   44.217461   -2.324420   38.781844    0.073586
DummyRegressor                      40.814142   -2.062626   37.223526    0.009371
LinearSVR                           28.957899   -1.150608   31.192565    0.016635
DecisionTreeRegressor               26.044443   -0.926496   29.522594    0.014799
OrthogonalMatchingPursuit           21.267356   -0.559027   26.558123    0.015085
LGBMRegressor                       20.999449   -0.538419   26.382008    0.087631
BaggingRegressor                    19.060760   -0.389289   25.070722    0.06

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:       17.3 | RMSE:       20.4 | MAPE:  3.46%
FLAML Test (catboost)          | MAE:       21.6 | RMSE:       27.4 | MAPE:  4.77%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       35.1 | RMSE:       44.4 | MAPE:  7.78%
SF AutoETS                     | MAE:       25.3 | RMSE:       31.1 | MAPE:  5.56%
SF SeasonalNaive               | MAE:       35.8 | RMSE:       43.1 | MAPE:  7.82%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model       MAE      RMSE     MAPE
     FLAML (catboost) 17.330279 20.352039 3.463134
FLAML Test (catboost) 21.575694 27.432342 4.772408
           SF AutoETS 25.260013 31.133410 5.558586
   Seasonal Naive (7) 26.800000 33.677907 5.792856
   Naive (Last Value) 34.007143 42.254307 7.509768
         SF AutoARIMA 35.115552 44.449183 7.783143
     SF SeasonalNaive 35.750000 43.130160 7.815577

Top 3:
                model       MAE      RMSE     MAPE
     FLAML (catboost) 17.330279 20.352039 3.463134
FLAML Test (catboost) 21.575694 27.432342 4.772408
           SF AutoETS 25.260013 31.133410 5.558586


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -20.82, Std: 17.86


## Interpretation and Business Insight

- Water demand peaks in summer due to outdoor use and irrigation
- Weekday demand is consistently higher than weekends
- Holiday periods show notable demand reduction
- Population growth creates a slow upward trend
- Temperature (not included) is the strongest external driver

## Limitations

1. Synthetic — real water demand depends on weather, temperature, and rainfall
2. No weather features
3. No distinction between residential, commercial, and industrial
4. Daily granularity — treatment plants need hourly forecasts

## How to Improve This Project

1. Add temperature and rainfall forecasts as features
2. Use hourly data for treatment plant operations
3. Segment by customer class and zone
4. Include pipe burst / leak detection anomaly signals
5. Model drought conditions separately

## Production Considerations

- Day-ahead forecasting for treatment plant scheduling
- Integration with SCADA systems
- Weather-adjusted forecast updates
- Reservoir level optimization dashboards

## Common Mistakes

1. Ignoring weather — temperature explains most summer peaks
2. Not accounting for water restrictions during droughts
3. Using annual averages for daily operational decisions
4. Ignoring pipe network losses (non-revenue water)

## Mini Challenge / Exercises

1. Add a synthetic temperature feature and measure correlation
2. Model weekend vs weekday demand separately
3. Detect anomalous demand days (possible pipe bursts)
4. Try monthly aggregation for infrastructure planning

## Final Summary / Key Takeaways

1. Water demand forecasting is critical for utility operations
2. Seasonal (summer) and weekly patterns are dominant
3. Weather is the key missing predictor
4. ML and statistical models both perform well here
5. Combining approaches gives robust forecasts for planning

In [18]:
import json
metrics = {
    'project': 'Water Demand Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Water Demand Forecasting — Complete ===")

Metrics saved.

=== Water Demand Forecasting — Complete ===
